In [ ]:
import pandas as pd
from pathlib import Path
import seaborn as sns
import holoviews as hv
import hvplot.pandas
import numpy as np

from workspace_dani.data_preprocessing.utils import data_handling
from workspace_dani.visualizations.donut_plots import make_donut_plot

hv.extension('bokeh')

In [ ]:
project_root_path = Path.cwd().parents[2]  #/ Path("data_preprocessing") / Path("utils")

labels_path = project_root_path / Path("intermediary_data") / Path("outcome_labels") / Path("sepsis") / Path(
    "sepsis_data_15min.parquet")
sofa_path = project_root_path / Path("intermediary_data") / Path("outcome_labels") / Path("sepsis") / Path(
    "sofa_data_15min.parquet")
si_path = project_root_path / Path("intermediary_data") / Path("outcome_labels") / Path("sepsis") / Path(
    "si_data_15min.parquet")

In [ ]:
df_sepsis = data_handling.load_parquet_table(parquet_path=labels_path, debug_mode=False, backend="numpy_nullable")
df_data_sofa = data_handling.load_parquet_table(parquet_path=sofa_path, debug_mode=False, backend="numpy_nullable")
df_data_si = data_handling.load_parquet_table(parquet_path=si_path, debug_mode=False, backend="numpy_nullable")
#df_data = df_data.convert_dtypes()

In [ ]:
df_sepsis.describe()

In [ ]:
df_sepsis["patientunitstayid"].unique()

In [ ]:
df_data_sofa.head()

In [ ]:
df_data_sofa.describe()

In [ ]:
df_data_si.describe()

### The labels for sepsis and suspected infections provide only the label "true", the associated label for missing data is "false"

In [ ]:
df_sepsis["patientunitstayid"].unique()

In [ ]:
df_sepsis.hvplot.scatter(x="labresultoffset", y="patientunitstayid")

In [ ]:
df_sepsis.iloc[0, :]

In [ ]:
sample_sepsis_df = df_sepsis[df_sepsis["patientunitstayid"] == 141194]

plot der für jeden Patienten das Delta zwischen den beiden Zeitpunkten illustriert

In [ ]:
sample_sofa_df = df_data_sofa[df_data_sofa["patientunitstayid"] == 141194]
sample_si_df = df_data_si[df_data_si["patientunitstayid"] == 141194]


In [ ]:
sample_sofa_df.head()

In [ ]:

sample_si_df.head()


In [ ]:
sample_sepsis_df.head()

In [ ]:
sample_sofa_df.hvplot(x="labresultoffset", y="sofa", label="sofa") * sample_si_df.hvplot(x="infusionoffset",
                                                                                         y="susp_inf",
                                                                                         label="sus infection") * sample_sepsis_df.hvplot(
    x="labresultoffset", y="sep3", kind="scatter", label="sepsis", color="magenta", size=100)

In [ ]:
df_sepsis.head()

In [ ]:
abs(sample_sepsis_df["abx_time"].values[0])

In [ ]:
sample_sepsis_df["abx_time"]

In [ ]:
sample_time = sample_sepsis_df["abx_time"] / np.timedelta64(1, "h")
sample_time.values[0]

In [ ]:
sample_sepsis_df["abx_time"].values[0].astype("timedelta64[m]")

In [ ]:
abs(sample_sepsis_df["abx_time"].values[0].astype("timedelta64[m]"))

In [ ]:
sample_sepsis_df["abx_time"].abs()

In [ ]:
abs(sample_sepsis_df["abx_time"])

In [ ]:
df_sepsis["timedelta"] = df_sepsis.apply(lambda x: x["labresultoffset"], axis=1)

In [ ]:
df_sepsis["timedelta"] = df_sepsis.apply(
    lambda x: abs(abs(x["labresultoffset"]) - abs(x["abx_time"])) if pd.isnull(x["samp_time"]) else abs(
        abs(x["labresultoffset"]) - abs(x["samp_time"])), axis=1)

## Taking a look at long time deltas, is there something wrong with them?


In [ ]:
df_sepsis["timedelta"].describe(percentiles=[0.75, 0.8, 0.9, 0.95])

In [ ]:
id_sepsis_label_biggest_td = df_sepsis.iloc[df_sepsis["timedelta"].idxmax(), :]["patientunitstayid"]
id_sepsis_label_biggest_td

In [ ]:
sepsis_btd = df_sepsis[df_sepsis["patientunitstayid"] == id_sepsis_label_biggest_td]
si_btd = df_data_si[df_data_si["patientunitstayid"] == id_sepsis_label_biggest_td]
sofa_btd = df_data_sofa[df_data_sofa["patientunitstayid"] == id_sepsis_label_biggest_td]


In [ ]:
from bokeh.models.formatters import DatetimeTickFormatter, BasicTickFormatter, NumeralTickFormatter

# formatter = DatetimeTickFormatter(months='%b')
formatter = DatetimeTickFormatter(months='%B')
#formatter = NumeralTickFormatter(format = "00:00:00")


In [ ]:
sofa_btd.hvplot(x="labresultoffset", y="sofa", label="sofa", xformatter=formatter) * si_btd.hvplot(x="infusionoffset",
                                                                                                   y="susp_inf",
                                                                                                   label="sus infection") * sepsis_btd.hvplot(
    x="labresultoffset", y="sep3", kind="scatter", label="sepsis", color="magenta", size=100)


In [ ]:
sepsis_timedelta_df = df_sepsis["timedelta"].dt.days
sepsis_timedelta_df.hvplot("timedelta", kind="hist", title="time delta in days distribution", bins=40)

In [ ]:
df_sepsis["timedelta"].quantile(0.95)

### This is how a prime wrong sepsis label looks like

In [ ]:

id_sepsis_label_2d_td = \
df_sepsis[df_sepsis["timedelta"] == df_sepsis["timedelta"].quantile(0.95)]["patientunitstayid"].iloc[2]
id_sepsis_label_2d_td

**iloc[0] looks good while iloc[1] seems to be way off**

In [ ]:
sepsis_2d = df_sepsis[df_sepsis["patientunitstayid"] == id_sepsis_label_2d_td]
si_2d = df_data_si[df_data_si["patientunitstayid"] == id_sepsis_label_2d_td]
sofa_2d = df_data_sofa[df_data_sofa["patientunitstayid"] == id_sepsis_label_2d_td]


In [ ]:
sofa_2d.hvplot(x="labresultoffset", y="sofa", label="sofa", xformatter=formatter) * si_2d.hvplot(x="infusionoffset",
                                                                                                 y="susp_inf",
                                                                                                 label="sus infection") * sepsis_2d.hvplot(
    x="labresultoffset", y="sep3", kind="scatter", label="sepsis", color="magenta", size=100)


import seaborn as sns

sns.jointplot(x=df_sepsis["labresultoffset"], y=df_sepsis["abx_time"], kind="hex", color="#4CB391")

In [ ]:
df_sepsis.hvplot("labresultoffset", "abx_time", kind="hexbin")

## Checking the number of sepsis labels in conjunction with possible cohorts

In [ ]:
admission_source_ids_path = project_root_path / Path(
    "intermediary_data/patient_unit_stay_ids/cohorts/admit_source_unit_type/included_ids_admit_source_unit_type.parquet")
admission_source_ids_path

In [ ]:
admission_source_inclusion_ids_df: pd.DataFrame = data_handling.load_parquet_table(
    parquet_path=admission_source_ids_path, debug_mode=False)
admission_source_inclusion_ids_df.head()

In [ ]:
df_sepsis_cohort = df_sepsis[
    df_sepsis["patientunitstayid"].isin(admission_source_inclusion_ids_df["Included ID"])]

#### We might need to adjust the cohorts, they're too restrictive, might need to consider the Borgwardt Cohort! 

In [ ]:
dataset_categorical_path = project_root_path / Path("intermediary_data") / Path("dataset") / Path("build") / Path(
    "2024-03-26") / Path("build_dataset_cats.parquet")

df_cats = data_handling.load_parquet_table(parquet_path=dataset_categorical_path, debug_mode=False)
df_cats.head()

In [ ]:
df_patients_with_sepsis_label: pd.DataFrame = df_cats[df_cats["patientunitstayid"].isin(df_sepsis["patientunitstayid"])]

In [ ]:
df_patients_with_sepsis_label["unittype"].value_counts()

In [ ]:
df_patients_with_sepsis_label["unitadmitsource"].value_counts()

In [ ]:
df_patients_with_sepsis_label["unitvisitnumber"].value_counts()

In [ ]:
unit_visit_number_upper_bound = 1
unit_admit_source_considered = ["Emergency Department", "Direct Admit"]

In [ ]:
df_sepsis_cohort_visit_number = df_patients_with_sepsis_label[df_patients_with_sepsis_label["unitvisitnumber"] <= 1]
df_sepsis_cohort_source = df_patients_with_sepsis_label[
    df_patients_with_sepsis_label["unitadmitsource"].isin(unit_admit_source_considered)]

In [ ]:
diagnosis_parquet_path: Path = project_root_path / Path(
    "intermediary_data/data_cleanup/basic_cleanup/2024-03-26/diagnosis.parquet")
df_diagnosis_table = data_handling.load_parquet_table(parquet_path=diagnosis_parquet_path, debug_mode=False)

### How does the number of detected Sepsis patients change with Sepsis 2 compared to Sepsis 3?

In [ ]:
df_diagnosis_table_sepsis_label = df_diagnosis_table[
    df_diagnosis_table["patientunitstayid"].isin(df_sepsis_cohort_source["patientunitstayid"])]

In [ ]:
df_sepsis_cohort_admission = df_sepsis_cohort_source[df_sepsis_cohort_source["apachedxgroup"] == "Sepsis"]
df_diagnosis_table_label_admission = df_diagnosis_table_sepsis_label[
    df_diagnosis_table_sepsis_label["patientunitstayid"].isin(df_sepsis_cohort_admission["patientunitstayid"])]

In [ ]:
sepsis_2_icd_code_condition: pd.Series = df_diagnosis_table_sepsis_label["icd9code"].astype(str).str.contains("A41",
                                                                                                              case=False)
sepsis_2_dx_name_condition: pd.Series = df_diagnosis_table_sepsis_label["diagnosisstring"].str.contains("sepsis",
                                                                                                        case=False)
df_sepsis_2_dx: pd.DataFrame = df_diagnosis_table_sepsis_label[sepsis_2_icd_code_condition | sepsis_2_dx_name_condition]


In [ ]:
df_sepsis_cohort_source["patientunitstayid"].unique()

In [ ]:
df_sepsis_2_dx["patientunitstayid"].unique()

### The number of sepsis detections with sepsis-2 is almost 3x lower than with the generated sepsis-3 labels !

In [ ]:
id_sample_with_blood_sampling = 3166218

In [ ]:
si_bs = df_data_si[df_data_si["patientunitstayid"] == id_sample_with_blood_sampling]
sofa_bs = df_data_sofa[df_data_sofa["patientunitstayid"] == id_sample_with_blood_sampling]
sepsis_bs = df_sepsis[df_sepsis["patientunitstayid"] == id_sample_with_blood_sampling]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
from bokeh.models.formatters import FuncTickFormatter, CustomJSTickFormatter

In [ ]:
formatter_code = """
var hours = Math.floor(tick / 3600000); // Convert milliseconds to hours
return hours + 'h';
"""

# Create the FuncTickFormatter with the custom JavaScript code
# formatter = FuncTickFormatter(code=formatter_code)
formatter = CustomJSTickFormatter(code=formatter_code)

In [ ]:
si_bs

In [ ]:
sepsis_bs

In [ ]:
sofa_bs.hvplot(x="labresultoffset", y="sofa", label="SOFA score", xformatter=formatter) * si_bs.hvplot(kind="scatter", x="infusionoffset",
                                                                           y="susp_inf",
                                                                           label="Suspected Infection", color="black",
                                                                           marker="s", size=100) * sepsis_bs.hvplot(
    x="labresultoffset", y="sep3", kind="scatter", label="Sepsis", color="magenta", marker="^", size=150)